# Summary Yahoo

Health check + coverage report for Yahoo Finance data in the local DuckDB.

Yahoo writes three tables: **`dividends`**, **`splits`** (corporate actions), and **`yahoo_prices`** (daily OHLCV).

Row-level retrieval goes through `irp.data.yahoo.prices()` / `dividends()` / `splits()`. Aggregations stay in SQL via the shared `db()` connection.

In [2]:
import pandas as pd
from IPython.display import display

from irp.data._common import db
from irp.data.yahoo import prices, dividends, splits

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## Tables

Schema, row count, and key stats for each table written by `irp.sources.yahoo.YahooSource`.

### `dividends`

Cash dividend events. One row per (Ticker, Date). Date is a `DATE` value (`YYYY-MM-DD`).

| Column | Type | Meaning |
|---|---|---|
| Ticker | VARCHAR | Uppercase symbol (e.g. `AAPL`) |
| Date | DATE | Ex-dividend date (e.g. `2024-01-02`) |
| Amount | DOUBLE | Dividend amount per share (in ticker's currency) |
| SrcId | VARCHAR | Source ticker (same as Ticker for Yahoo) |
| Src | VARCHAR | Loader name; always `yahoo` |

In [3]:
_sample_div = dividends(tickers='AAPL')
display(_sample_div.dtypes.to_frame('dtype'))
display(_sample_div.tail())

,dtype
Ticker,str
Date,int64
Amount,float64
SrcId,str
Src,str


,Ticker,Date,Amount,SrcId,Src
86,AAPL,20250512,0.26,AAPL,yahoo
87,AAPL,20250811,0.26,AAPL,yahoo
88,AAPL,20251110,0.26,AAPL,yahoo
89,AAPL,20260209,0.26,AAPL,yahoo
90,AAPL,20260511,0.27,AAPL,yahoo


In [4]:
_stats_div = db().execute("""
    SELECT
        COUNT(*)               AS rows,
        COUNT(DISTINCT Ticker) AS tickers,
        MIN(Date)              AS date_min,
        MAX(Date)              AS date_max
    FROM dividends
""").df().T
_stats_div.columns = ['dividends']
display(_stats_div)

,dividends
rows,395940
tickers,6947
date_min,19620116
date_max,20260515


### `splits`

Stock split events. One row per (Ticker, Date). Date is a `DATE` value (`YYYY-MM-DD`).

| Column | Type | Meaning |
|---|---|---|
| Ticker | VARCHAR | Uppercase symbol |
| Date | DATE | Split effective date (e.g. `2024-01-02`) |
| Ratio | DOUBLE | Split ratio (e.g. `4.0` = 4-for-1 split) |
| SrcId | VARCHAR | Source ticker |
| Src | VARCHAR | Always `yahoo` |

In [5]:
_sample_spl = splits(tickers='AAPL')
display(_sample_spl.dtypes.to_frame('dtype'))
display(_sample_spl.tail())

,dtype
Ticker,str
Date,int64
Ratio,float64
SrcId,str
Src,str


,Ticker,Date,Ratio,SrcId,Src
0,AAPL,19870616,2.0,AAPL,yahoo
1,AAPL,20000621,2.0,AAPL,yahoo
2,AAPL,20050228,2.0,AAPL,yahoo
3,AAPL,20140609,7.0,AAPL,yahoo
4,AAPL,20200831,4.0,AAPL,yahoo


In [6]:
_stats_spl = db().execute("""
    SELECT
        COUNT(*)               AS rows,
        COUNT(DISTINCT Ticker) AS tickers,
        MIN(Date)              AS date_min,
        MAX(Date)              AS date_max
    FROM splits
""").df().T
_stats_spl.columns = ['splits']
display(_stats_spl)

,splits
rows,8610
tickers,3253
date_min,19621031
date_max,20260514


### `yahoo_prices`

Daily OHLCV bars. One row per (Ticker, Date). Prices are auto-adjusted (splits + dividends) by yfinance. Date is a `DATE` value (`YYYY-MM-DD`).

| Column | Type | Meaning |
|---|---|---|
| Ticker | VARCHAR | Uppercase symbol |
| Date | DATE | Bar date (e.g. `2024-01-02`) |
| Open / High / Low / Close | DOUBLE | Auto-adjusted OHLC |
| Volume | BIGINT | Daily volume |

In [7]:
_sample_yp = prices(tickers='AAPL', start='2025-04-01')
display(_sample_yp.dtypes.to_frame('dtype'))
display(_sample_yp.tail())

,dtype
Ticker,str
Date,int64
Open,float64
High,float64
Low,float64
Close,float64
Volume,int64


,Ticker,Date,Open,High,Low,Close,Volume
278,AAPL,20260511,291.980011,293.880005,290.230011,292.679993,42247300
279,AAPL,20260512,292.559998,295.269989,292.559998,294.799988,45748100
280,AAPL,20260513,293.500000,300.920013,293.500000,298.869995,52684300
281,AAPL,20260514,299.820007,300.450012,295.380005,298.209991,35324900
282,AAPL,20260515,297.899994,303.200012,296.519989,300.230011,54721100


In [8]:
_stats_yp = db().execute("""
    SELECT
        COUNT(*)                     AS rows,
        COUNT(DISTINCT Ticker)       AS tickers,
        MIN(Date)                    AS date_min,
        MAX(Date)                    AS date_max,
        COUNT(DISTINCT Date)         AS distinct_dates
    FROM yahoo_prices
""").df().T
_stats_yp.columns = ['yahoo_prices']
display(_stats_yp)

,yahoo_prices
rows,34623556
tickers,11524
date_min,19271230
date_max,20260518
distinct_dates,24962


## Cross-table coverage

Overlap between corporate actions, prices, and company metadata.

In [9]:
display(db().execute("""
    SELECT
        COUNT(DISTINCT d.Ticker)                                          AS dividend_tickers,
        COUNT(DISTINCT s.Ticker)                                          AS split_tickers,
        COUNT(DISTINCT p.Ticker)                                          AS price_tickers,
        COUNT(DISTINCT p.Ticker) FILTER (WHERE c.Ticker IS NOT NULL)      AS prices_with_company,
        COUNT(DISTINCT p.Ticker) FILTER (WHERE c.Ticker IS NULL)          AS prices_without_company,
        COUNT(DISTINCT d.Ticker) FILTER (WHERE p.Ticker IS NULL)          AS dividends_no_prices,
        COUNT(DISTINCT s.Ticker) FILTER (WHERE p.Ticker IS NULL)          AS splits_no_prices
    FROM yahoo_prices p
    FULL OUTER JOIN dividends  d ON p.Ticker = d.Ticker
    FULL OUTER JOIN splits     s ON p.Ticker = s.Ticker
    LEFT JOIN      companies   c ON p.Ticker = c.Ticker
""").df().T.rename(columns={0: 'count'}))
print('Tickers without company metadata: non-equity instruments and equities not covered by SimFin.')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,count
dividend_tickers,6947
split_tickers,3253
price_tickers,11524
prices_with_company,4066
prices_without_company,7458
dividends_no_prices,0
splits_no_prices,0


Tickers without company metadata: non-equity instruments and equities not covered by SimFin.


## Freshness — when was data last updated?

Latest date per table. `yahoo_prices` freshness by market (via Stooq markets mapping).

In [10]:
display(db().execute("""
    SELECT 'dividends'    AS tbl, MAX(Date) AS latest_date, COUNT(*) AS rows, COUNT(DISTINCT Ticker) AS tickers FROM dividends
    UNION ALL
    SELECT 'splits'       AS tbl, MAX(Date) AS latest_date, COUNT(*) AS rows, COUNT(DISTINCT Ticker) AS tickers FROM splits
    UNION ALL
    SELECT 'yahoo_prices' AS tbl, MAX(Date) AS latest_date, COUNT(*) AS rows, COUNT(DISTINCT Ticker) AS tickers FROM yahoo_prices
    ORDER BY tbl
""").df())

,tbl,latest_date,rows,tickers
0,dividends,20260515,395940,6947
1,splits,20260514,8610,3253
2,yahoo_prices,20260518,34623556,11524


In [11]:
display(db().execute("""
    SELECT
        m.Market,
        MAX(p.Date)              AS latest_date,
        COUNT(DISTINCT p.Ticker) AS tickers,
        COUNT(*)                 AS rows
    FROM yahoo_prices p
    LEFT JOIN universe m ON p.Ticker = m.Ticker
    GROUP BY m.Market
    ORDER BY latest_date DESC
""").df())

,Market,latest_date,tickers,rows
0,indices,20260518,15,110634
1,nasdaq etfs,20260515,937,1414060
2,nysemkt stocks,20260515,283,1271285
3,nyse stocks,20260515,3269,13379952
4,nasdaq stocks,20260515,4468,12308181
5,cryptocurrencies,20260515,52,241308
6,nyse etfs,20260515,2552,6139444


## Quality

Quick checks on `yahoo_prices`. For full anomaly review use `notebooks/review_stooq_anomalies.ipynb` (Stooq vs Yahoo divergence).

In [12]:
display(db().execute("""
    SELECT
        COUNT(*) FILTER (WHERE Open   IS NULL) AS missing_Open,
        COUNT(*) FILTER (WHERE High   IS NULL) AS missing_High,
        COUNT(*) FILTER (WHERE Low    IS NULL) AS missing_Low,
        COUNT(*) FILTER (WHERE Close  IS NULL) AS missing_Close,
        COUNT(*) FILTER (WHERE Volume IS NULL) AS missing_Volume
    FROM yahoo_prices
""").df())

,missing_Open,missing_High,missing_Low,missing_Close,missing_Volume
0,7,7,7,7,0


In [13]:
_neg_tickers = db().execute(
    'SELECT DISTINCT Ticker FROM yahoo_prices WHERE Close < 0 OR Open < 0 OR High < 0 OR Low < 0'
).df()['Ticker'].tolist()
if _neg_tickers:
    print(f'Tickers with negative prices: {len(_neg_tickers)}')
    print(_neg_tickers)
else:
    print('No negative-price rows.')

Tickers with negative prices: 4
['SAFE', 'VATE', 'VHI', 'CBIO']


In [14]:
display(db().execute("""
    SELECT Ticker, COUNT(*) AS rows
    FROM yahoo_prices
    WHERE High < Low OR Close > High OR Close < Low
    GROUP BY Ticker
    ORDER BY rows DESC
    LIMIT 20
""").df())

,Ticker,rows
0,VHI,4318
1,VATE,2635
2,CBIO,2189
3,JOB,1031
4,OPY,963
5,GTY,908
6,WLYB,889
7,UG,874
8,HTO,862
9,RMCF,848


## Dividends — top payers

In [15]:
display(db().execute("""
    SELECT
        Ticker,
        COUNT(*)              AS events,
        MIN(Date)             AS first_date,
        MAX(Date)             AS last_date,
        ROUND(AVG(Amount), 4) AS avg_amount
    FROM dividends
    GROUP BY Ticker
    ORDER BY events DESC
    LIMIT 20
""").df())

,Ticker,events,first_date,last_date,avg_amount
0,PPH,494,20000211,20260401,0.0703
1,PBT,472,19861224,20260430,0.0625
2,SBR,472,19870109,20260515,0.2581
3,DNP,472,19870223,20260430,0.0656
4,NCA,469,19871209,20260515,0.0416
5,MMT,469,19870508,20260414,0.0497
6,CIK,468,19870526,20260515,0.0425
7,CMU,467,19870424,20260414,0.0349
8,MFM,467,19870209,20260414,0.0422
9,NNY,466,19871008,20260515,0.0414


## Splits — most active

In [16]:
display(db().execute("""
    SELECT
        Ticker,
        COUNT(*)  AS events,
        MIN(Date) AS first_date,
        MAX(Date) AS last_date
    FROM splits
    GROUP BY Ticker
    ORDER BY events DESC
    LIMIT 20
""").df())

,Ticker,events,first_date,last_date
0,UBFO,38,19970123,20170405
1,SNFCA,33,19890105,20250711
2,TR,29,19860711,20260305
3,CBSH,27,19860527,20251202
4,ADM,26,19801027,20010830
5,AROW,25,19851002,20220916
6,GTY,24,19780106,19970401
7,FNRN,22,19990224,20260227
8,VLY,21,19920427,20120509
9,LARK,20,20011212,20251201
